# Phase 5: Reranking

This notebook:
- Creates training data from labeled query-product pairs
- Trains XGBoost reranker (LambdaMART-style)
- Evaluates reranking performance
- Saves the trained model

## 5.1 Setup

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import time
from tqdm import tqdm

from src.retrieval import HybridRetriever
from src.query_understanding import QueryUnderstanding
from src.reranker import Reranker, FeatureExtractor, create_training_data_from_labels

print("✓ Imports successful")

## 5.2 Load Components

In [ ]:
# Load retriever
retriever = HybridRetriever(indices_dir='../data/indices')

In [ ]:
# Load query understanding
qu = QueryUnderstanding(use_cache=True)
print(f"✓ QueryUnderstanding loaded (cache: {len(qu.cache)} entries)")

In [ ]:
# Load product data
df_products = pd.read_parquet('../data/processed/products.parquet')

# Filter to indexed products only
indexed_ids = set(retriever.product_ids)
df_products = df_products[df_products['product_id'].isin(indexed_ids)].reset_index(drop=True)

print(f"✓ Loaded {len(df_products):,} products")

In [ ]:
# Load labeled data
df_train_labels = pd.read_parquet('../data/processed/labels_train.parquet')
df_test_labels = pd.read_parquet('../data/processed/labels_test.parquet')

# Filter to products in our index
df_train_labels = df_train_labels[df_train_labels['product_id'].isin(indexed_ids)]
df_test_labels = df_test_labels[df_test_labels['product_id'].isin(indexed_ids)]

print(f"✓ Train labels: {len(df_train_labels):,} pairs ({df_train_labels['query'].nunique():,} queries)")
print(f"✓ Test labels: {len(df_test_labels):,} pairs ({df_test_labels['query'].nunique():,} queries)")

In [ ]:
# Check label distribution
print("\nLabel Distribution (Train):")
print(df_train_labels['esci_label'].value_counts())

## 5.3 Create Training Data

In [ ]:
# Create training data (this retrieves candidates for each query)
print("Creating training data...")
print("(This may take a few minutes)\n")

train_data = create_training_data_from_labels(
    labels_df=df_train_labels,
    retriever=retriever,
    query_understanding=None,
    max_queries=500,
    candidates_per_query=50
)

print(f"\n✓ Created {len(train_data)} training queries")
print(f"  Total samples: {sum(len(d['labels']) for d in train_data):,}")

In [ ]:
# Create validation data
print("Creating validation data...")

val_data = create_training_data_from_labels(
    labels_df=df_test_labels,
    retriever=retriever,
    query_understanding=None,
    max_queries=200,
    candidates_per_query=50
)

print(f"\n✓ Created {len(val_data)} validation queries")
print(f"  Total samples: {sum(len(d['labels']) for d in val_data):,}")

In [ ]:
# Check label distribution in training data
all_labels = [l for d in train_data for l in d['labels']]
print("\nLabel distribution in training data:")
print(pd.Series(all_labels).value_counts().sort_index())
print("\n0=Irrelevant, 1=Complement, 2=Substitute, 3=Exact")

## 5.4 Train Reranker

In [ ]:
# Initialize reranker with product data
reranker = Reranker(product_df=df_products)
print("✓ Reranker initialized")

In [ ]:
# Train the model
print("Training reranker...\n")

reranker.train(
    train_data=train_data,
    val_data=val_data,
    params={
        'objective': 'rank:ndcg',
        'learning_rate': 0.1,
        'max_depth': 6,
        'n_estimators': 100,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'random_state': 42,
        'n_jobs': -1
    }
)

## 5.5 Test Reranking

In [ ]:
# Create product lookup for display
product_lookup = df_products.set_index('product_id').to_dict('index')

def display_comparison(query, hybrid_results, reranked_results, top_k=5):
    """Display before/after reranking."""
    print(f"\nQuery: '{query}'")
    print("=" * 70)
    
    print("\nBefore Reranking (Hybrid):")
    for i, r in enumerate(hybrid_results[:top_k]):
        title = product_lookup.get(r['product_id'], {}).get('product_title', 'N/A')[:50]
        print(f"  {i+1}. {title}...")
    
    print("\nAfter Reranking:")
    for i, r in enumerate(reranked_results[:top_k]):
        title = product_lookup.get(r['product_id'], {}).get('product_title', 'N/A')[:50]
        orig_rank = r.get('original_rank', '?')
        print(f"  {i+1}. (was #{orig_rank}) {title}...")

In [ ]:
# Test queries
test_queries = [
    "ceramic mugs bulk",
    "nike running shoes",
    "organic candles lavender",
    "iphone case",
    "eco friendly bags wholesale"
]

for query in test_queries:
    # Get hybrid results
    hybrid_results = retriever.search(query, method="hybrid", top_k=50)
    
    # Rerank
    reranked_results = reranker.rerank(query, hybrid_results, top_k=10)
    
    display_comparison(query, hybrid_results, reranked_results)

## 5.6 Latency Analysis

In [ ]:
query = "ceramic mugs wholesale"
n_runs = 10

# Measure hybrid only
times_hybrid = []
for _ in range(n_runs):
    start = time.time()
    results = retriever.search(query, method="hybrid", top_k=50)
    times_hybrid.append((time.time() - start) * 1000)

# Measure reranking only
times_rerank = []
results = retriever.search(query, method="hybrid", top_k=50)
for _ in range(n_runs):
    start = time.time()
    reranker.rerank(query, results, top_k=20)
    times_rerank.append((time.time() - start) * 1000)

# Measure full pipeline
times_full = []
for _ in range(n_runs):
    start = time.time()
    results = retriever.search(query, method="hybrid", top_k=50)
    reranked = reranker.rerank(query, results, top_k=20)
    times_full.append((time.time() - start) * 1000)

print("Latency Analysis (ms):")
print("-" * 45)
print(f"Hybrid retrieval:  mean={np.mean(times_hybrid):.1f}, p95={np.percentile(times_hybrid, 95):.1f}")
print(f"Reranking only:    mean={np.mean(times_rerank):.1f}, p95={np.percentile(times_rerank, 95):.1f}")
print(f"Full pipeline:     mean={np.mean(times_full):.1f}, p95={np.percentile(times_full, 95):.1f}")

## 5.7 Save Model

In [ ]:
# Save the trained reranker
reranker.save('../data/models/reranker.pkl')

In [ ]:
# Verify we can load it
reranker_loaded = Reranker(
    model_path='../data/models/reranker.pkl',
    product_df=df_products
)

# Quick test
test_results = retriever.search("ceramic mugs", method="hybrid", top_k=10)
reranked = reranker_loaded.rerank("ceramic mugs", test_results, top_k=5)
print(f"✓ Model loads and works! Got {len(reranked)} results.")

## 5.8 Summary

In [ ]:
print("\n" + "=" * 60)
print("PHASE 5 COMPLETE")
print("=" * 60)

print(f"""
Reranking Summary
-----------------

Model: XGBoost Ranker (LambdaMART-style)
Training queries: {len(train_data)}
Validation queries: {len(val_data)}

Features used:
  - Hybrid score, rank position
  - Source flags (BM25, FAISS, both)
  - Query-title overlap
  - Brand match, wholesale signal match
  - Product completeness

Pipeline:
  Query -> Hybrid Search (100) -> Rerank -> Top 20

Files:
  - src/reranker.py (Reranker class)
  - data/models/reranker.pkl (trained model)

Next: Run 06_evaluation.ipynb (Full evaluation with metrics)
""")